# CarDD YOLO Segmentation — Experiment 4

מחברת זו ממשיכה ישירות מהניסוי הטוב של טל, במקום להתחיל שוב מ־`yolov8s-seg.pt`.

## השינויים המרכזיים

1. התחלה ממשקלי `best.pt` של טל.
2. אימון ברזולוציה `1024`.
3. Fine-tuning עם learning rate נמוך יותר ו־cosine decay.
4. oversampling מתון של תמונות שמכילות `crack`, המחלקה החלשה ביותר.
5. אוגמנטציות גאומטריות מתונות שמתאימות לנזקים קטנים ודקים.
6. הפחתת Mosaic לקראת סוף האימון באמצעות `close_mosaic`.
7. הערכת mAP עם `conf=0.001`, ולא עם `conf=0.5`.
8. השוואה אוטומטית מול המודל של טל על אותו Validation Set.

> המחברת אינה משנה את ה־Validation Set ואינה משתמשת ב־Test Set לצורך בחירת המודל.


In [1]:
# התקנה — להריץ פעם אחת בלבד בסביבה חדשה
# %pip install -U ultralytics pandas matplotlib pyyaml pillow opencv-python


In [2]:
from pathlib import Path
import os
import json
import random
import shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports completed")


Imports completed


## 1. הגדרות נתיבים

In [3]:
# ============================================================
# USER CONFIGURATION — לשנות רק את הנתיבים כאן
# ============================================================

DATASET_ROOT = Path(r"/tf/DATAMAN/Damage_part")

TRAIN_IMAGES_DIR = DATASET_ROOT / "images" / "train"
TRAIN_LABELS_DIR = DATASET_ROOT / "labels" / "train"
VAL_IMAGES_DIR = DATASET_ROOT / "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

# משקלי המודל הטוב של טל — exp3_hsv_tuned
TAL_BEST_MODEL = Path(
    r"/tf/DATAMAN/Damage_part/runs/segment/car_damage_yolo_seg/exp3_hsv_tuned/weights/best.pt"
)

PROJECT_DIR = Path("runs/segment/car_damage_yolo_seg")
EXP_NAME = "exp4_tal_finetune_balanced"

BALANCED_TRAIN_LIST = DATASET_ROOT / "train_exp4_balanced.txt"
EXP4_DATA_YAML = DATASET_ROOT / "data_exp4.yaml"

CLASS_NAMES = {
    0: "dent",
    1: "scratch",
    2: "crack",
    3: "glass shatter",
    4: "lamp broken",
    5: "tire flat",
}

for path, label in [
    (TRAIN_IMAGES_DIR, "train images"),
    (TRAIN_LABELS_DIR, "train labels"),
    (VAL_IMAGES_DIR, "validation images"),
    (VAL_LABELS_DIR, "validation labels"),
    (TAL_BEST_MODEL, "Tal best model"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

print("All paths exist")


All paths exist


## 2. בדיקת הדאטה

הבדיקה סופרת מופעים לפי מחלקה ומוודאת שלכל תמונה יש קובץ label מתאים.


In [4]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(folder: Path):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def read_yolo_seg_classes(label_path: Path):
    classes = []
    if not label_path.exists():
        return classes
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
        parts = line.strip().split()
        if not parts:
            continue
        if len(parts) < 7:
            raise ValueError(
                f"Invalid segmentation label in {label_path}, line {line_number}: "
                f"expected class + at least 3 polygon points"
            )
        cls = int(float(parts[0]))
        if cls not in CLASS_NAMES:
            raise ValueError(f"Unknown class {cls} in {label_path}")
        classes.append(cls)
    return classes

train_images = image_files(TRAIN_IMAGES_DIR)
val_images = image_files(VAL_IMAGES_DIR)

missing_train_labels = [
    p.name for p in train_images
    if not (TRAIN_LABELS_DIR / f"{p.stem}.txt").exists()
]
missing_val_labels = [
    p.name for p in val_images
    if not (VAL_LABELS_DIR / f"{p.stem}.txt").exists()
]

print(f"Train images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")
print(f"Missing train labels: {len(missing_train_labels)}")
print(f"Missing validation labels: {len(missing_val_labels)}")

if missing_train_labels or missing_val_labels:
    raise RuntimeError("Some images do not have matching label files")


Train images: 2852
Validation images: 810
Missing train labels: 0
Missing validation labels: 0


In [5]:
train_instance_counts = Counter()
train_images_per_class = Counter()

image_classes = {}

for image_path in train_images:
    label_path = TRAIN_LABELS_DIR / f"{image_path.stem}.txt"
    classes = read_yolo_seg_classes(label_path)
    image_classes[image_path] = classes
    train_instance_counts.update(classes)
    train_images_per_class.update(set(classes))

distribution_df = pd.DataFrame([
    {
        "class_id": cls,
        "class_name": CLASS_NAMES[cls],
        "instances": train_instance_counts[cls],
        "images_containing_class": train_images_per_class[cls],
    }
    for cls in CLASS_NAMES
])

display(distribution_df)


,class_id,class_name,instances,images_containing_class
0,0,dent,1806,1242
1,1,scratch,2560,1507
2,2,crack,651,434
3,3,glass shatter,475,469
4,4,lamp broken,494,489
5,5,tire flat,225,219


## 3. יצירת רשימת אימון מאוזנת

המודל של טל היה חלש במיוחד ב־`crack`. לכן המחברת מוסיפה את תמונות ה־crack פעם נוספת לרשימת האימון.

זו אינה העתקה ל־Validation או ל־Test, אלא sampling חוזר מתוך ה־Train בלבד.

ה־oversampling מוגבל ל־פי 2 כדי לא לגרום ל־overfitting חריף למחלקה הקטנה.


In [6]:
CRACK_CLASS_ID = 2
CRACK_REPEAT_FACTOR = 2  # כל תמונת crack תופיע פעמיים בסך הכול

balanced_paths = list(train_images)
crack_images = [
    image_path
    for image_path, classes in image_classes.items()
    if CRACK_CLASS_ID in classes
]

for _ in range(CRACK_REPEAT_FACTOR - 1):
    balanced_paths.extend(crack_images)

random.shuffle(balanced_paths)

BALANCED_TRAIN_LIST.write_text(
    "\n".join(str(p.resolve()) for p in balanced_paths) + "\n",
    encoding="utf-8",
)

print(f"Original train images: {len(train_images)}")
print(f"Images containing crack: {len(crack_images)}")
print(f"Balanced training entries: {len(balanced_paths)}")
print(f"Saved list: {BALANCED_TRAIN_LIST}")


Original train images: 2852
Images containing crack: 434
Balanced training entries: 3286
Saved list: /tf/DATAMAN/Damage_part/train_exp4_balanced.txt


In [7]:
yaml_content = {
    "path": str(DATASET_ROOT.resolve()),
    "train": str(BALANCED_TRAIN_LIST.resolve()),
    "val": str((DATASET_ROOT / "images" / "val").resolve()),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

with open(EXP4_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_content, f, sort_keys=False, allow_unicode=True)

print(EXP4_DATA_YAML.read_text(encoding="utf-8"))


path: /tf/DATAMAN/Damage_part
train: /tf/DATAMAN/Damage_part/train_exp4_balanced.txt
val: /tf/DATAMAN/Damage_part/images/val
nc: 6
names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat



## 4. ניקוי cache

כאשר משנים את רשימת האימון, חשוב למחוק קובצי cache ישנים.


In [8]:
for cache_file in DATASET_ROOT.rglob("*.cache"):
    cache_file.unlink()
    print("Deleted:", cache_file)


## 5. אימון Exp4

### למה הפרמטרים שונים מהניסוי של טל?

- `model=TAL_BEST_MODEL`: ממשיכים ממודל שכבר למד את CarDD.
- `imgsz=1024`: מעט יותר פרטים עבור שריטות וסדקים דקים.
- `lr0=0.001`: Fine-tuning עדין יותר ממשקלים קיימים.
- `cos_lr=True`: הורדה הדרגתית של קצב הלמידה.
- `close_mosaic=15`: סוף האימון מתבצע ללא Mosaic כדי לחדד גבולות מסכה.
- `degrees`, `translate`, `scale`: שונות גאומטרית מתונה.
- HSV מתון יותר מהניסוי האגרסיבי של טל כדי לשמור סימני נזק עדינים.
- `copy_paste=0.0`: לא מדביקים נזק על חלק רכב אקראי, משום שזה עלול ליצור הקשר לא מציאותי.


In [9]:
model_exp4 = YOLO(str(TAL_BEST_MODEL))

results_exp4 = model_exp4.train(
    data=str(EXP4_DATA_YAML),
    epochs=70,
    imgsz=1024,
    batch=4,             # להעלות ל-8 רק אם יש מספיק VRAM
    device=0,
    workers=0,

    patience=25,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    # אוגמנטציות מתונות
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    shear=1.0,
    perspective=0.0002,
    fliplr=0.5,
    flipud=0.0,

    mosaic=0.50,
    close_mosaic=15,
    mixup=0.0,
    copy_paste=0.0,

    # שמירה ובקרת ניסוי
    seed=SEED,
    deterministic=True,
    amp=True,
    cache=False,
    plots=True,
    save=True,
    project=str(PROJECT_DIR),
    name=EXP_NAME,
    exist_ok=False,
)

print("Training completed")
print("Best weights:", Path(results_exp4.save_dir) / "weights" / "best.pt")


New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.99 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24049MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tf/DATAMAN/Damage_part/data_exp4.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.55, hsv_v=0.35, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tf/DATAMAN/Damage_

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 6.0it/s 17.1s0.2s
                   all        810       1744      0.773       0.66      0.713      0.526      0.751      0.667        0.7      0.515

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      22/70      5.75G     0.9061      1.605     0.8269      1.247          0          7       1024: 100% ━━━━━━━━━━━━ 822/822 5.0it/s 2:440.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 102/102 6.1it/s 16.8s0.2s
                   all        810       1744      0.748      0.685      0.715      0.525      0.741      0.675      0.702      0.523

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      23/70      3.88G     0.89

OutOfMemoryError: CUDA out of memory. Tried to allocate 54.00 MiB. GPU 0 has a total capacity of 23.48 GiB of which 30.00 MiB is free. Process 107232 has 0 bytes memory in use. Process 126901 has 0 bytes memory in use. Process 128014 has 0 bytes memory in use. Process 137858 has 0 bytes memory in use. Including non-PyTorch memory, this process has 0 bytes memory in use. Of the allocated memory 5.67 GiB is allocated by PyTorch, and 90.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [12]:
!nvidia-smi

Wed Aug  5 12:27:51 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        On  | 00000000:03:00.0 Off |                  N/A |
|  0%   38C    P8              24W / 370W |   6313MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
import os
import signal
import subprocess

current_pid = os.getpid()
print("Current Jupyter kernel PID:", current_pid)

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-compute-apps=pid",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
)

gpu_pids = []

for line in result.stdout.splitlines():
    line = line.strip()
    if line.isdigit():
        gpu_pids.append(int(line))

print("GPU processes:", gpu_pids)

for pid in gpu_pids:
    if pid == current_pid:
        print(f"Keeping current notebook kernel: {pid}")
        continue

    try:
        os.kill(pid, signal.SIGKILL)
        print(f"Killed process: {pid}")
    except ProcessLookupError:
        print(f"Process {pid} already ended")
    except PermissionError:
        print(f"No permission to kill process: {pid}")

In [13]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [14]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

2.632723331451416 GB allocated
3.509765625 GB reserved


In [19]:
import os

print("התיקייה הנוכחית:")
print(os.getcwd())

התיקייה הנוכחית:
/tf/Noa_new/Damages/Experiments


In [20]:
from pathlib import Path

matches = list(Path(".").rglob("last.pt"))

print(f"נמצאו {len(matches)} קבצים:")
for path in matches:
    print(path.resolve())

נמצאו 4 קבצים:
/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced/weights/last.pt
/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights/last.pt
/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-3/weights/last.pt
/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-2/weights/last.pt


In [ ]:
from ultralytics import YOLO

model = YOLO("/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights/last.pt")

results = model.train(
    resume=True,
    batch=2
)

החזרה לזכרון של הנקודת סיום האחרונה והמשך אימון ממנה

In [29]:
from pathlib import Path

p = Path("/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights/last.pt")

print(p.exists())
print(p)

True
/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights/last.pt


In [30]:
from ultralytics import YOLO

model = YOLO(
    "/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/"
    "car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights/last.pt"
)

print("Checkpoint loaded successfully")

Checkpoint loaded successfully


In [32]:
print("epoch:", model.ckpt.get("epoch"))
print("optimizer exists:", model.ckpt.get("optimizer") is not None)

epoch: -1
optimizer exists: False


In [33]:
from pathlib import Path

weights_dir = Path(
    "/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/"
    "car_damage_yolo_seg/exp4_tal_finetune_balanced-4/weights"
)

for f in weights_dir.glob("*.pt"):
    print(f.name)

best.pt
last.pt


In [34]:
import pandas as pd

results_csv = (
    "/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/"
    "car_damage_yolo_seg/exp4_tal_finetune_balanced-4/results.csv"
)

df = pd.read_csv(results_csv)

print(df[["epoch"]].tail())
print("Last completed epoch:", df["epoch"].iloc[-1])

    epoch
65     66
66     67
67     68
68     69
69     70
Last completed epoch: 70


המודל כן התאמן 70 אפוקים וסיים את האימון שלו- אין צורך להמשיך

The training stopped during Epoch 33 because of a `CUDA Out Of Memory` error. After freeing GPU memory, the training was continued and completed up to Epoch 70, as shown in `results.csv`, but the later output was not saved in the notebook.



## 6. עקומות אימון ובחירת epoch

המחברת בודקת את מדדי המסכה, לא רק את מדדי ה־bounding box.


In [39]:
from pathlib import Path

run_dir = Path(
    "/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/"
    "car_damage_yolo_seg/exp4_tal_finetune_balanced-4"
)

results_csv = run_dir / "results.csv"

print(results_csv)
print(results_csv.exists())

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

metric_cols = [
    "metrics/precision(M)",
    "metrics/recall(M)",
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)",
]

display(df[["epoch"] + metric_cols].tail())

best_idx = df["metrics/mAP50-95(M)"].idxmax()
print("Best epoch:", int(df.loc[best_idx, "epoch"]))
print("Best mask mAP50-95:", float(df.loc[best_idx, "metrics/mAP50-95(M)"]))

/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/results.csv
True


,epoch,metrics/precision(M),metrics/recall(M),metrics/mAP50(M),metrics/mAP50-95(M)
65,66,0.81201,0.69475,0.73387,0.54762
66,67,0.80849,0.69222,0.73250,0.54815
67,68,0.84956,0.67372,0.73230,0.54753
68,69,0.84046,0.68221,0.73575,0.54883
69,70,0.82543,0.68932,0.73261,0.54700


Best epoch: 69
Best mask mAP50-95: 0.54883


In [40]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

pairs = [
    ("train/seg_loss", "val/seg_loss", "Segmentation loss"),
    ("metrics/precision(M)", None, "Mask precision"),
    ("metrics/recall(M)", None, "Mask recall"),
    ("metrics/mAP50-95(M)", None, "Mask mAP50-95"),
]

for ax, (col1, col2, title) in zip(axes.flat, pairs):
    ax.plot(df["epoch"], df[col1], label=col1)
    if col2:
        ax.plot(df["epoch"], df[col2], label=col2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


<Figure size 1400x900 with 4 Axes>

## 7. Validation תקין לצורך mAP

לא משתמשים ב־`conf=0.5` לחישוב mAP.  
סף נמוך מאפשר לבנות את עקומת Precision–Recall על פני טווח רחב של confidences.


In [41]:
EXP4_BEST = run_dir / "weights" / "best.pt"
exp4_best = YOLO(str(EXP4_BEST))

metrics_exp4 = exp4_best.val(
    data=str(EXP4_DATA_YAML),
    split="val",
    imgsz=1024,
    conf=0.001,
    iou=0.70,
    max_det=100,
    batch=4,
    device=0,
    workers=0,
    plots=True,
    project=str(PROJECT_DIR),
    name=f"{EXP_NAME}_val",
    exist_ok=True,
)

print("Validation output:", metrics_exp4.save_dir)
print("Mask mAP50:", metrics_exp4.seg.map50)
print("Mask mAP50-95:", metrics_exp4.seg.map)
print("Mask precision:", metrics_exp4.seg.mp)
print("Mask recall:", metrics_exp4.seg.mr)


Ultralytics 8.4.99 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24049MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,781,922 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5121.0±222.1 MB/s, size: 690.3 KB)
val: Scanning /tf/DATAMAN/Damage_part/labels/val.cache... 810 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 810/810 154.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 203/203 7.5it/s 26.9s0.1ss
                   all        810       1744      0.764      0.724      0.736      0.564      0.838      0.681      0.733      0.548
                  dent        352        501       0.65      0.517      0.531      0.289      0.752      0.442      0.517      0.272
               scratch        431        728      0.565      0.534      0.501      0.278      0.688      0.482       0.51      0.244
              

## 8. השוואה הוגנת מול המודל של טל

שני המודלים מוערכים מחדש עם אותם פרמטרים:
- אותו Validation Set
- אותה רזולוציה
- אותו confidence
- אותו IoU/NMS


In [42]:
tal_model = YOLO(str(TAL_BEST_MODEL))

metrics_tal = tal_model.val(
    data=str(EXP4_DATA_YAML),
    split="val",
    imgsz=1024,
    conf=0.001,
    iou=0.70,
    max_det=100,
    batch=4,
    device=0,
    workers=0,
    plots=False,
    verbose=False,
)

comparison_rows = []

for cls_idx, class_name in CLASS_NAMES.items():
    tal_ap = float(metrics_tal.seg.maps[cls_idx])
    exp4_ap = float(metrics_exp4.seg.maps[cls_idx])
    comparison_rows.append({
        "class_id": cls_idx,
        "class_name": class_name,
        "Tal_AP50_95": tal_ap,
        "Exp4_AP50_95": exp4_ap,
        "delta": exp4_ap - tal_ap,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

summary_df = pd.DataFrame([
    {
        "model": "Tal",
        "mask_precision": float(metrics_tal.seg.mp),
        "mask_recall": float(metrics_tal.seg.mr),
        "mask_mAP50": float(metrics_tal.seg.map50),
        "mask_mAP50_95": float(metrics_tal.seg.map),
    },
    {
        "model": "Exp4",
        "mask_precision": float(metrics_exp4.seg.mp),
        "mask_recall": float(metrics_exp4.seg.mr),
        "mask_mAP50": float(metrics_exp4.seg.map50),
        "mask_mAP50_95": float(metrics_exp4.seg.map),
    },
])

display(summary_df)
comparison_df.to_csv(run_dir / "comparison_vs_tal_per_class.csv", index=False)
summary_df.to_csv(run_dir / "comparison_vs_tal_overall.csv", index=False)


Ultralytics 8.4.99 🚀 Python-3.11.13 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24049MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,781,922 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5239.4±138.2 MB/s, size: 712.0 KB)
val: Scanning /tf/DATAMAN/Damage_part/labels/val.cache... 810 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 810/810 161.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 203/203 6.3it/s 32.4s0.2ss
                   all        810       1744      0.787       0.72       0.75      0.572        0.8      0.704       0.74      0.551
Speed: 0.4ms preprocess, 5.9ms inference, 0.0ms loss, 1.5ms postprocess per image


,class_id,class_name,Tal_AP50_95,Exp4_AP50_95,delta
0,0,dent,0.283886,0.271657,-0.012229
1,1,scratch,0.266452,0.243950,-0.022503
2,2,crack,0.203422,0.212534,0.009112
3,3,glass shatter,0.886149,0.898085,0.011936
4,4,lamp broken,0.745517,0.753048,0.007530
5,5,tire flat,0.918539,0.907355,-0.011183


,model,mask_precision,mask_recall,mask_mAP50,mask_mAP50_95
0,Tal,0.800025,0.704458,0.739641,0.550661
1,Exp4,0.837959,0.681230,0.733395,0.547772


## 9. בחירת המודל

המודל החדש נחשב שיפור רק אם:

1. `mask mAP50-95` הכללי גבוה יותר.
2. ביצועי `crack` משתפרים באופן משמעותי.
3. אין ירידה חריפה ב־`glass shatter`, `lamp broken` או `tire flat`.
4. התמונות הוויזואליות מראות גבולות מסכה סבירים ולא רק שיפור מספרי קטן.


In [43]:
overall_delta = (
    float(metrics_exp4.seg.map) -
    float(metrics_tal.seg.map)
)

crack_delta = float(
    comparison_df.loc[
        comparison_df["class_name"] == "crack",
        "delta"
    ].iloc[0]
)

print(f"Overall mAP50-95 delta: {overall_delta:+.4f}")
print(f"Crack AP50-95 delta: {crack_delta:+.4f}")

if overall_delta > 0 and crack_delta > 0:
    print("Candidate accepted for external test-set evaluation.")
else:
    print("Do not replace Tal's model yet. Review per-class regressions and visuals.")


Overall mAP50-95 delta: -0.0029
Crack AP50-95 delta: +0.0091
Do not replace Tal's model yet. Review per-class regressions and visuals.


## 10. דוגמאות ויזואליות מה־Validation Set


In [44]:
val_sample = random.sample(
    [str(p) for p in val_images],
    min(12, len(val_images))
)

predictions = exp4_best.predict(
    source=val_sample,
    imgsz=1024,
    conf=0.25,
    iou=0.70,
    device=0,
    save=True,
    project=str(PROJECT_DIR),
    name=f"{EXP_NAME}_visuals",
    exist_ok=True,
    verbose=False,
)

print("Saved prediction examples")


Results saved to /tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced_visuals
Saved prediction examples


## 11. יצוא המודל

רק לאחר שהמודל החדש מנצח ב־Validation, מריצים עליו פעם אחת את סקריפט ההערכה החיצוני על Test Set של CarDD.


In [45]:
EXPORT_PATH = Path("damage_YOLO_model_EXP4.pt")

if overall_delta > 0 and crack_delta > 0:
    shutil.copy2(EXP4_BEST, EXPORT_PATH)
    print("Exported:", EXPORT_PATH.resolve())
else:
    print("Model was not exported because it did not pass the selection rule.")


Model was not exported because it did not pass the selection rule.


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
run_dir = Path(
    "/tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/"
    "car_damage_yolo_seg/exp4_tal_finetune_balanced-4"
)

per_class_csv = run_dir / "comparison_vs_tal_per_class.csv"
overall_csv = run_dir / "comparison_vs_tal_overall.csv"

# ------------------------------------------------------------
# Load saved comparison results
# ------------------------------------------------------------
comparison_df = pd.read_csv(per_class_csv)
summary_df = pd.read_csv(overall_csv)

print("Per-class comparison:")
display(comparison_df)

print("Overall comparison:")
display(summary_df)

# ------------------------------------------------------------
# Prepare values
# ------------------------------------------------------------
class_names = [
    "Dent",
    "Scratch",
    "Crack",
    "Glass Shatter",
    "Lamp Broken",
    "Tire Flat",
]

tal_values = comparison_df["Tal_AP50_95"].to_numpy()
noa_values = comparison_df["Exp4_AP50_95"].to_numpy()

tal_overall = float(
    summary_df.loc[
        summary_df["model"] == "Tal",
        "mask_mAP50_95"
    ].iloc[0]
)

noa_overall = float(
    summary_df.loc[
        summary_df["model"] == "Exp4",
        "mask_mAP50_95"
    ].iloc[0]
)

# Add Overall as final category
labels = class_names + ["Overall"]
tal_values = np.append(tal_values, tal_overall)
noa_values = np.append(noa_values, noa_overall)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
x = np.arange(len(labels))
width = 0.36

plt.figure(figsize=(12, 6))

bars1 = plt.bar(
    x - width/2,
    tal_values,
    width,
    label="Tal – Exp3 Tuned HSV"
)

bars2 = plt.bar(
    x + width/2,
    noa_values,
    width,
    label="Noa – Exp4 Fine-Tuned"
)

plt.xlabel("Damage Class")
plt.ylabel("Mask mAP50–95")
plt.title("Validation Mask mAP50–95: Tal vs Noa")

plt.xticks(x, labels, rotation=20)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis="y", alpha=0.25)

# Values above bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.015,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.tight_layout()

save_path = run_dir / "tal_vs_noa_validation_map50_95.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved to:", save_path)

Per-class comparison:


,class_id,class_name,Tal_AP50_95,Exp4_AP50_95,delta
0,0,dent,0.283886,0.271657,-0.012229
1,1,scratch,0.266452,0.243950,-0.022503
2,2,crack,0.203422,0.212534,0.009112
3,3,glass shatter,0.886149,0.898085,0.011936
4,4,lamp broken,0.745517,0.753048,0.007530
5,5,tire flat,0.918539,0.907355,-0.011183


Overall comparison:


,model,mask_precision,mask_recall,mask_mAP50,mask_mAP50_95
0,Tal,0.800025,0.704458,0.739641,0.550661
1,Exp4,0.837959,0.681230,0.733395,0.547772


<Figure size 1200x600 with 1 Axes>

Saved to: /tf/Noa_new/Damages/Experiments/runs/segment/runs/segment/car_damage_yolo_seg/exp4_tal_finetune_balanced-4/tal_vs_noa_validation_map50_95.png
